# Azure Coding Agent - Interactive Notebook

This notebook provides an interactive interface to the coding agent with support for both **Local LLM (LM Studio)** and **Azure OpenAI**.

## Setup

Make sure you have:
1. Installed dependencies: `pip install -r requirements.txt`
2. For **Local LLM**: LM Studio running with your model loaded
3. For **Azure OpenAI**: `.env` file with Azure credentials
4. Selected the correct Python kernel (your venv)

## Choose Your Backend:
- **Local LLM** (Free, Private): Use with LM Studio - recommended for deepseek/deepseek-r1-0528-qwen3-8b
- **Azure OpenAI** (Cloud): Use with Azure GPT-4/3.5

In [1]:
# Import the agent and utility functions
from azure_agent import (
    AzureCodeAgent,
    run,
    load_history,
    save_history,
    clear_history,
    view_history,
    force_save
)

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

print("✅ Imports successful!")

✅ Imports successful!


## Option 1: Quick Start - Run with prompt.md

Choose your backend below. This will read the project spec from `prompt.md` and execute it automatically.

### Option 1A: Local LLM (LM Studio) - Recommended
Make sure LM Studio is running with your model loaded!

In [ ]:
# Run with Local LLM (DeepSeek model)
run(
    use_local_llm=True,                        # Use local LLM
    local_model="qwen/qwen3-coder-30b",  # Your model
    local_api_base="http://localhost:1234/v1", # LM Studio default
    context_window=8000,
    max_tokens=2000,
    temperature=0.7,
    continue_from_history=True,                # Continue from previous session
    inactivity_timeout=300,                    # Stop after 5 minutes of inactivity
    auto_save_interval=30                      # Save every 30 seconds
)

🏠 Using Local LLM
   API Base: http://localhost:1234/v1
   Model: qwen/qwen3-coder-30b
🆕 Starting fresh session...
🔄 Auto-save enabled (every 30 seconds)
🐕 Watchdog enabled (inactivity timeout: 300s)
STARTING AZURE AGENT - Watch the output below
🐕 Inactivity timeout: 300 seconds (resets on activity)

Iteration 1/50


### Option 1B: Azure OpenAI (Cloud)
Only run this if you have Azure OpenAI configured in .env

In [ ]:
# Run with Azure OpenAI (requires Azure credentials in .env)
# run(
#     use_local_llm=False,                   # Use Azure OpenAI
#     continue_from_history=True,
#     inactivity_timeout=300,
#     auto_save_interval=30
# )

## Option 2: Manual Control - Create Agent and Chat

For more control over the conversation.

Choose your backend:

In [ ]:
# Create agent with Local LLM (DeepSeek)
agent = AzureCodeAgent(
    use_local_llm=True,
    local_model="deepseek/deepseek-r1-0528-qwen3-8b",
    max_tokens=2000,
    temperature=0.7
)

print(f"✅ Agent created!")
print(f"📍 Working directory: {agent.working_directory}")
print(f"🤖 Model: {agent.model_name}")

# For Azure OpenAI instead, uncomment:
# agent = AzureCodeAgent()  # Uses env vars from .env

In [ ]:
# Send a custom message to the agent
from pathlib import Path

# Read the project spec
prompt_file = Path("prompt.md")
prompt_text = prompt_file.read_text(encoding="utf-8")

# Chat with the agent
response = agent.chat(f"""
We are in the project directory: {agent.working_directory}

Here is the project specification you must implement:

--- BEGIN SPEC (prompt.md) ---
{prompt_text}
--- END SPEC ---

Read the spec carefully, then:
1) Plan the tasks.
2) Implement the project step by step in this directory.
3) Show commands and file changes as you go.
""", display=True)

In [ ]:
# Continue the conversation with a follow-up message
agent.chat("Continue from where you left off. Complete any remaining tasks.", display=True)

In [ ]:
# Send a custom instruction
agent.chat("Add unit tests for all the functions you created.", display=True)

## History Management

In [ ]:
# View conversation history summary
view_history()

In [ ]:
# Save current conversation
force_save(agent)

In [ ]:
# Load previous history into agent
previous_messages = load_history()
if previous_messages:
    agent.messages = previous_messages
    print(f"📚 Loaded {len(previous_messages)} previous messages")

In [ ]:
# Clear history to start fresh
clear_history()

## Advanced Usage

In [ ]:
# Create agent with custom settings

# Option A: Local LLM with custom settings
custom_agent = AzureCodeAgent(
    use_local_llm=True,
    local_model="deepseek/deepseek-r1-0528-qwen3-8b",
    local_api_base="http://localhost:1234/v1",
    context_window=8000,
    max_tokens=2000,
    temperature=0.5  # Lower = more focused
)

# Option B: Azure OpenAI with custom settings (commented out)
# custom_agent = AzureCodeAgent(
#     api_key="your-api-key",  # Or use env var
#     azure_endpoint="https://your-resource.openai.azure.com/",
#     deployment_name="gpt-4",
#     api_version="2024-02-15-preview"
# )

# Set custom working directory
from pathlib import Path
custom_agent.working_directory = project_dir  # Or any other path

# Set max iterations
custom_agent.max_iterations = 100

print("✅ Custom agent created!")
print(f"🤖 Model: {custom_agent.model_name}")

In [ ]:
# Execute specific bash command directly
result = agent.execute_bash_command("ls -la")
print("Output:", result['stdout'])
print("Success:", result['success'])

In [ ]:
# Read a file directly
file_result = agent.read_file("prompt.md")
if file_result['success']:
    print("File content:")
    print(file_result['content'])

In [ ]:
# Write a file directly
write_result = agent.write_file(
    "test.txt",
    "This is a test file created by the agent."
)
print(write_result['message'])

## Inspect Agent State

In [ ]:
# View current message count
print(f"📊 Messages in conversation: {len(agent.messages)}")

# View last message
if agent.messages:
    last_msg = agent.messages[-1]
    print(f"\n📝 Last message ({last_msg['role']}):")
    print(last_msg['content'][:500] + "..." if len(last_msg['content']) > 500 else last_msg['content'])

In [ ]:
# View agent configuration
print(f"🔧 Agent Configuration:")
print(f"  Mode: {'Local LLM' if agent.use_local_llm else 'Azure OpenAI'}")
if agent.use_local_llm:
    print(f"  API Base: {agent.local_api_base}")
    print(f"  Model: {agent.model_name}")
    print(f"  Context Window: {agent.context_window}")
    print(f"  Max Tokens: {agent.max_tokens}")
else:
    print(f"  Endpoint: {agent.azure_endpoint}")
    print(f"  Deployment: {agent.deployment_name}")
print(f"  Working Dir: {agent.working_directory}")
print(f"  Max Iterations: {agent.max_iterations}")
print(f"  Temperature: {agent.temperature}")

## Examples

In [ ]:
# Example 1: Create a simple Python script
agent = AzureCodeAgent(
    use_local_llm=True,
    local_model="deepseek/deepseek-r1-0528-qwen3-8b"
)

agent.chat("""
Create a Python script called 'hello.py' that:
1. Accepts a name as a command-line argument
2. Prints a personalized greeting
3. If no name is provided, prints a generic greeting
""", display=True)

In [ ]:
# Example 2: Data processing task
agent = AzureCodeAgent(
    use_local_llm=True,
    local_model="deepseek/deepseek-r1-0528-qwen3-8b"
)

agent.chat("""
Create a Python script that:
1. Generates a CSV file with random data (100 rows, 3 columns: name, age, score)
2. Reads the CSV and calculates average score
3. Finds the person with the highest score
4. Saves the results to a text file
""", display=True)

In [ ]:
# Example 3: Web scraping setup
agent = AzureCodeAgent(
    use_local_llm=True,
    local_model="deepseek/deepseek-r1-0528-qwen3-8b"
)

agent.chat("""
Set up a simple web scraping project:
1. Create requirements.txt with necessary packages (requests, beautifulsoup4)
2. Create a script that fetches a webpage and extracts all links
3. Add error handling
4. Create a README with usage instructions
""", display=True)

## Tips

1. **Start Fresh**: Run `clear_history()` before starting a new project
2. **Resume Work**: Use `continue_from_history=True` to pick up where you left off
3. **Monitor Progress**: Set `display=True` to see real-time output
4. **Save Often**: Use `force_save(agent)` to manually save important states
5. **Local LLM**: Use `use_local_llm=True` with your DeepSeek model (free and private!)
6. **Azure OpenAI**: Use GPT-3.5 deployment for testing, GPT-4 for production (costs money)
7. **Timeout Protection**: Always set `inactivity_timeout` to prevent runaway execution

## Troubleshooting

### Local LLM Issues:
- **Connection refused**: Make sure LM Studio server is running (http://localhost:1234)
- **Slow responses**: Use a smaller model or reduce `context_window` and `max_tokens`
- **Model not found**: Check the exact model name in LM Studio
- **Out of memory**: Reduce `context_window` to 4096 or lower

### General Issues:
- **Module not found**: Make sure you selected the correct kernel (your venv)
- **Azure errors**: Check your `.env` file has correct credentials
- **Agent stuck**: Interrupt the cell and check `agent_history.json`
- **Permission denied**: Agent can only work in its `working_directory`

## Local LLM vs Azure OpenAI

| Feature | Local LLM | Azure OpenAI |
|---------|-----------|--------------|
| **Cost** | Free | Pay per token |
| **Privacy** | Fully local | Cloud-based |
| **Speed** | Depends on GPU | Fast & consistent |
| **Setup** | LM Studio + model | Azure account |
| **Offline** | ✅ Works offline | ❌ Needs internet |

**Recommended**: Start with local LLM (free), upgrade to Azure if you need GPT-4 quality!